In [0]:
%run ./00_config

# Scratch / Practice Notebook

This notebook is my personal scratch space for experimenting and practicing new PySpark/Spark SQL concepts while building this project.

**It is NOT part of the graded pipeline.** The real pipeline is: `00_config` -> `01_setup_landing_zone` -> `02_bronze_ingestion` -> `03_silver_cleaning` -> `04_gold_aggregations` -> `05_analytics` -> `06_incremental_test`.

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
file_name = f"yellow_tripdata_{NEW_MONTH}.csv.gz"
source = f"{SOURCE_PATH}/{file_name}"
target = f"{LANDING_PATH}/{file_name}"

dbutils.fs.cp(source, target)
print("Copied:", file_name)

Copied: yellow_tripdata_2019-04.csv.gz


In [0]:
bronze_before = spark.table(BRONZE_TABLE).count()
silver_before = spark.table(SILVER_TABLE).count()

print("Bronze count BEFORE:", bronze_before)
print("Silver count BEFORE:", silver_before)

Bronze count BEFORE: 50769555
Silver count BEFORE: 50169175


In [0]:
bronze_after = spark.table(BRONZE_TABLE).count()
print("Bronze count AFTER:", bronze_after)
print("New rows added:", bronze_after - bronze_before)

Bronze count AFTER: 50769555
New rows added: 0


In [0]:
silver_after = spark.table(SILVER_TABLE).count()
print("Silver count AFTER:", silver_after)
print("New rows added:", silver_after - silver_before)

Silver count AFTER: 50169175
New rows added: 0


In [0]:
display(spark.sql(f"SELECT SUM(total_trips) AS total FROM {GOLD_DEMAND_TABLE}"))

total
50169175


## Incremental Load Proof (REQ-05)

Simulated a new month (April 2019) arriving after the pipeline was already built and run on Jan-Mar.

| Layer  | Before     | After      | New rows added |
|--------|------------|------------|-----------------|
| Bronze | 22,519,712 | 29,952,851 | 7,433,139       |
| Silver | 22,277,544 | 29,630,378 | 7,352,834       |
| Gold   | (recomputed from Silver) | 29,630,378 (matches Silver exactly) |

**What this proves:**
- Auto Loader + checkpoints correctly detected and processed ONLY the new file - no reprocessing, no duplicates in Bronze or Silver.
- Silver's cleaning rate on the new data (about 1.08% removed) matched the original Jan-Mar rate almost exactly, confirming the R6 rule (pickup month must match the file's own month) generalizes correctly to any new month, not just a hardcoded date range.
- Gold tables fully recomputed from Silver and now reflect all 4 months combined, proving the whole pipeline - not just one layer - handles new data correctly.

In [0]:
file_name = "yellow_tripdata_2019-05.csv.gz"
source = f"{SOURCE_PATH}/{file_name}"
target = f"{LANDING_PATH}/{file_name}"

dbutils.fs.cp(source, target)
print("Copied:", file_name)

Copied: yellow_tripdata_2019-05.csv.gz


In [0]:
print("Bronze:", spark.table(BRONZE_TABLE).count())
print("Silver:", spark.table(SILVER_TABLE).count())

Bronze: 50769555
Silver: 50169175


In [0]:
months_to_add = ["2019-06", "2019-07"]

for month in months_to_add:
    file_name = f"yellow_tripdata_{month}.csv.gz"
    source = f"{SOURCE_PATH}/{file_name}"
    target = f"{LANDING_PATH}/{file_name}"
    dbutils.fs.cp(source, target)
    print("Copied:", file_name)

Copied: yellow_tripdata_2019-06.csv.gz
Copied: yellow_tripdata_2019-07.csv.gz


In [0]:
print("Bronze:", spark.table(BRONZE_TABLE).count())
print("Silver:", spark.table(SILVER_TABLE).count())

Bronze: 50769555
Silver: 50169175


## Extended Incremental Proof - Future Months (May, June, July)

To confirm the pipeline isn't hardcoded to April specifically, two more rounds were tested with zero code changes.

**Round 1: May alone**

| Layer  | Before     | After      | New rows added |
|--------|------------|------------|-----------------|
| Bronze | 29,952,851 | 37,518,112 | 7,565,261       |
| Silver | 29,630,378 | 37,107,957 | 7,477,579       |

**Round 2: June + July together (dropped in at the same time)**

| Layer  | Before     | After      | New rows added |
|--------|------------|------------|-----------------|
| Bronze | 37,518,112 | 50,769,555 | 13,251,443      |
| Silver | 37,107,957 | 50,169,175 | 13,061,218      |

**What this proves:**
- The pipeline correctly processed a single new month (May) with no code changes - confirming the design generalizes beyond the original April test.
- The pipeline also correctly handled TWO new files arriving at once (June + July), proving Auto Loader isn't limited to one new file per run.
- Cleaning removal rates stayed consistent across all rounds (around 1.1 percent to 1.4 percent), confirming the Silver rules (especially the dynamic file-month check) generalize correctly to any new month.
- No duplicate rows were created in any round, and no manual code edits were needed between rounds - only new files were dropped into the landing zone.